## Variables y Paqueterías

In [ ]:
# Periodos
AñoMes = 202608   # Carpeta de entrada (ej. 202512). None = la más reciente
AñoMes_Output  = None     # ANOMES del archivo de salida. None = igual que la carpeta
AñoMes_ant     = 202607   # Periodo anterior con información. None = mes anterior a AñoMes

# Rutas base (único lugar donde se definen)
RUTA_MARINE           = r"C:\Users\IKAL14\Documents\Integral\Marine"
RUTA_INTEGRAL_INSUMOS = r"C:\Users\IKAL14\Documents\Integral\Insumos"

# Paqueterías
import os
import re
from copy import copy
from pathlib import Path

import pandas as pd
import openpyxl
import xlrd
import win32com.client
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter

# Resolver AñoMes = None -> carpeta YYYYMM más reciente en Contabilidad
if AñoMes is None:
    _base = os.path.join(RUTA_INTEGRAL_INSUMOS, "Contabilidad")
    _folders = sorted(f for f in os.listdir(_base) if re.fullmatch(r"\d{6}", f))
    if not _folders:
        raise FileNotFoundError(f"No se encontraron carpetas YYYYMM en {_base}")
    AñoMes = int(_folders[-1])

# Periodo anterior: si es None se deriva del mes anterior; si no, se valida
if AñoMes_ant is None:
    _y, _m = divmod(AñoMes, 100)
    AñoMes_ant = (_y - 1) * 100 + 12 if _m == 1 else AñoMes - 1
if AñoMes_ant >= AñoMes:
    raise ValueError(f"AñoMes_ant ({AñoMes_ant}) debe ser menor que AñoMes ({AñoMes})")

ANIO = AñoMes // 100


## Conversión de Archivos

In [ ]:
carpetas = [
    os.path.join(RUTA_MARINE, "Insumos", str(AñoMes)),
    os.path.join(RUTA_MARINE, "Insumos", "Legacy"),
    os.path.join(RUTA_INTEGRAL_INSUMOS, "Contabilidad", str(AñoMes)),
]

def convertir_carpeta(excel, carpeta):
    if not os.path.isdir(carpeta):
        print(f"⚠ No existe la carpeta: {carpeta}")
        return
    for archivo in os.listdir(carpeta):
        if archivo.endswith(".xls") and not archivo.endswith(".xlsx"):
            ruta_completa = os.path.join(carpeta, archivo)
            wb = None
            try:
                wb = excel.Workbooks.Open(ruta_completa)
                nueva_ruta = ruta_completa + "x"
                wb.SaveAs(nueva_ruta, FileFormat=51)
                print(f"✓ Convertido: {archivo}")
            except Exception as e:
                print(f"✗ Error con {archivo}: {e}")
            finally:
                if wb is not None:
                    wb.Close(SaveChanges=False)

excel = win32com.client.Dispatch("Excel.Application")
excel.Visible = False
excel.DisplayAlerts = False

try:
    for carpeta in carpetas:
        print(f"\n--- Procesando: {carpeta} ---")
        convertir_carpeta(excel, carpeta)
finally:
    excel.Quit()

print("\nConversión terminada, todo en segundo plano")


--- Procesando: C:\Users\IKAL14\Documents\Integral\Marine\Insumos\202608 ---

--- Procesando: C:\Users\IKAL14\Documents\Integral\Marine\Insumos\Legacy ---

--- Procesando: C:\Users\IKAL14\Documents\Integral\Insumos\Contabilidad\202608 ---

Conversión terminada, todo en segundo plano


## Generar payments

In [ ]:

# ───────────────────────────────────────────────────────────────────────────────


BASE_DIR         = Path(RUTA_INTEGRAL_INSUMOS) / "Contabilidad"
SHEET_CANDIDATES = [str(ANIO), "KOTS-V"]


def get_folder(AñoMes):
    if AñoMes:
        folder = BASE_DIR / AñoMes
        if not folder.exists():
            raise FileNotFoundError(f"No existe la carpeta: {folder}")
        return folder
    folders = sorted([f for f in BASE_DIR.iterdir() if f.is_dir() and re.fullmatch(r"\d{6}", f.name)])
    if not folders:
        raise FileNotFoundError(f"No se encontraron carpetas YYYYMM en {BASE_DIR}")
    return folders[-1]


def extract_tab_name(filepath):
    stem = filepath.stem
    match = re.search(r"\bAP\s+(\w+)\s+vf$", stem, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    match = re.search(r"(\w+)\s+vf$", stem, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return stem


def get_source_sheet(filepath, data_only=False, read_only=False):
    """Abre el archivo y devuelve la hoja del año (SHEET_CANDIDATES) o KOTS-V."""
    
    if filepath.suffix.lower() == ".xls":
        xls_book = xlrd.open_workbook(filepath)
        found_sheet = None
        
        for candidate in SHEET_CANDIDATES:
            for i in range(xls_book.nsheets):
                if xls_book.sheet_by_index(i).name == candidate:
                    found_sheet = xls_book.sheet_by_index(i)
                    break
            if found_sheet:
                break
        
        if not found_sheet:
            found_sheet = xls_book.sheet_by_index(0)
        
        new_wb = Workbook()
        new_ws = new_wb.active
        new_ws.title = found_sheet.name
        
        for row_idx in range(found_sheet.nrows):
            for col_idx in range(found_sheet.ncols):
                value = found_sheet.cell_value(row_idx, col_idx)
                new_ws.cell(row_idx + 1, col_idx + 1).value = value
        
        return new_ws
    else:
        wb = openpyxl.load_workbook(filepath, data_only=data_only, read_only=read_only)
        
        for candidate in SHEET_CANDIDATES:
            if candidate in wb.sheetnames:
                return wb[candidate]
        
        first = wb.sheetnames[0]
        print(f"  [aviso] Hoja {SHEET_CANDIDATES} no encontrada, usando '{first}'")
        return wb[first]


def find_header_row(ws):
    """Detecta encabezados buscando palabras clave tipicas."""
    keywords = ['kots', 'acc voucher', 'payment','due date','period','policy n°','profit center','share','cost center', 'invoice', 'account', 'ceding', 'date']
    
    for row_idx in range(1, min(50, ws.max_row + 1)):
        row_values = [str(ws.cell(row_idx, col).value).lower() 
                      for col in range(1, ws.max_column + 1) 
                      if ws.cell(row_idx, col).value is not None]
        
        keyword_count = sum(1 for kw in keywords if any(kw in val for val in row_values))
        
        if keyword_count >= 7:
            return row_idx
    
    for row_idx in range(1, min(50, ws.max_row + 1)):
        row_data = [ws.cell(row_idx, col).value for col in range(1, ws.max_column + 1)]
        non_empty = len([x for x in row_data if x is not None])
        if non_empty > 5:
            return row_idx
    
    return 1


def copy_cell_style(from_cell, to_cell):
    """Copia el estilo de una celda a otra."""
    if from_cell.has_style:
        to_cell.font = copy(from_cell.font)
        to_cell.border = copy(from_cell.border)
        to_cell.fill = copy(from_cell.fill)
        to_cell.number_format = copy(from_cell.number_format)
        to_cell.protection = copy(from_cell.protection)
        to_cell.alignment = copy(from_cell.alignment)


def copy_range_with_format(from_ws, to_ws, start_row, end_row, from_ws_values=None):
    """Copia un rango preservando formato. Valores de from_ws_values (data_only) si está disponible."""
    for row_idx in range(start_row, end_row + 1):
        for col_idx in range(1, from_ws.max_column + 1):
            from_cell = from_ws.cell(row_idx, col_idx)
            to_row = row_idx - start_row + 1
            to_cell = to_ws.cell(to_row, col_idx)
            
            # Usa valores calculados (matriz precargada) si está disponible, sino usa directamente
            if from_ws_values is not None:
                fila = from_ws_values[row_idx - 1] if row_idx - 1 < len(from_ws_values) else ()
                to_cell.value = fila[col_idx - 1] if col_idx - 1 < len(fila) else None
            else:
                to_cell.value = from_cell.value
            
            copy_cell_style(from_cell, to_cell)


def copy_column_widths(from_ws, to_ws):
    """Copia los anchos de columna."""
    for col_idx in range(1, from_ws.max_column + 1):
        col_letter = get_column_letter(col_idx)
        if from_ws.column_dimensions[col_letter].width:
            to_ws.column_dimensions[col_letter].width = from_ws.column_dimensions[col_letter].width

def clean_header_values(ws):
    """Limpia los encabezados de espacios raros, saltos de línea y caracteres especiales."""
    # Recolecta todas las posiciones de "Amount USD"
    amount_usd_positions = []
    
    for col_idx in range(1, ws.max_column + 1):
        header_cell = ws.cell(1, col_idx)
        if header_cell.value is not None:
            value = str(header_cell.value)
            # Reemplaza saltos de línea con espacios
            value = value.replace('\n', ' ').replace('\r', ' ')
            # Reemplaza múltiples espacios con uno solo
            value = ' '.join(value.split())
            # Quita espacios al inicio y final
            value = value.strip()
            
            # Guarda posición si es "Amount USD"
            if value == "Amount USD":
                amount_usd_positions.append(col_idx)
            
            header_cell.value = value
    
    # Renombra solo la ÚLTIMA "Amount USD" (la más a la derecha)
    if amount_usd_positions:
        last_col = amount_usd_positions[-1]
        ws.cell(1, last_col).value = "Amount USD (CORRECT)"

folder = get_folder(str(AñoMes))
anomes = str(AñoMes_Output) if AñoMes_Output else folder.name

print(f"Carpeta de entrada : {folder}")
print(f"AñoMes del output  : {anomes}")

excel_files = [
    f for f in folder.iterdir()
    if f.suffix.lower() in (".xlsx", ".xls", ".xlsm")
    and not f.name.startswith("~$")
    and re.search(r"\bAP\s+\w+\s+vf", f.stem, re.IGNORECASE)
    # Omitir .xls que ya fueron convertidos a .xlsx (celda de conversión)
    and not (f.suffix.lower() == ".xls" and f.with_suffix(".xlsx").exists())
]

if not excel_files:
    raise FileNotFoundError(f"No se encontraron archivos 'AP * vf' en {folder}")

print(f"Archivos encontrados: {[f.name for f in excel_files]}\n")

output_path = folder / f"Payments_{anomes}.xlsx"
output_wb = openpyxl.Workbook()
output_wb.remove(output_wb.active)

for filepath in sorted(excel_files):
    tab_name = extract_tab_name(filepath)
    print(f"  Leyendo '{filepath.name}' -> pestaña '{tab_name}' ...", end=" ")
    
    # Lee dos veces: con formato y con valores calculados
    source_ws = get_source_sheet(filepath, data_only=False)  # Con formato
    # Valores en modo read_only, volcados una sola vez a una matriz (evita acceso celda por celda)
    ws_values = get_source_sheet(filepath, data_only=True, read_only=True)
    values_grid = [list(r) for r in ws_values.iter_rows(values_only=True)]
    ws_values.parent.close()  # libera el archivo (read_only lo mantiene abierto)
    
    header_row = find_header_row(source_ws)
    end_row = source_ws.max_row
    
    num_rows = end_row - header_row + 1
    
    output_ws = output_wb.create_sheet(tab_name)
    
    copy_range_with_format(source_ws, output_ws, header_row, end_row, values_grid)
    copy_column_widths(source_ws, output_ws)
    clean_header_values(output_ws)  # Limpia los encabezados
    source_ws.parent.close()        # libera el archivo con formato
    
    print(f"({num_rows} filas, desde fila {header_row})")

output_wb.save(output_path)
print(f"\nArchivo generado : {output_path}")

Carpeta de entrada : C:\Users\IKAL14\Documents\Integral\Insumos\Contabilidad\202608
ANOMES_CARPETA del output  : 202608
Archivos encontrados: ['260831 2026 08 AP AE vf.xlsx', '260831 2026 08 AP CL vf.xlsx', '260831 2026 08 AP VA vf.xlsx']

  Leyendo '260831 2026 08 AP AE vf.xlsx' -> pestaña 'AE' ... 

(362 filas, desde fila 21)
  Leyendo '260831 2026 08 AP CL vf.xlsx' -> pestaña 'CL' ... (393 filas, desde fila 23)
  Leyendo '260831 2026 08 AP VA vf.xlsx' -> pestaña 'VA' ... (928 filas, desde fila 24)

Archivo generado : C:\Users\IKAL14\Documents\Integral\Insumos\Contabilidad\202608\Payments_202608.xlsx


## OSLR_KOT_INTEGRAL

In [ ]:
# =============================================================================
# Configuración: periodo, rutas y BDX
# =============================================================================
mes = AñoMes % 100

MESES_ES = {1: 'ENERO', 2: 'FEBRERO', 3: 'MARZO', 4: 'ABRIL', 5: 'MAYO', 6: 'JUNIO',
            7: 'JULIO', 8: 'AGOSTO', 9: 'SEPTIEMBRE', 10: 'OCTUBRE', 11: 'NOVIEMBRE', 12: 'DICIEMBRE'}
str1_month = MESES_ES[mes]            # TODO EN MAYÚSCULAS
str2_month = str1_month.capitalize()  # Primera en mayúscula

directorio_proyecto  = Path(RUTA_MARINE).as_posix()
directorio_proyecto2 = Path(RUTA_INTEGRAL_INSUMOS).as_posix()

# Rutas de los BDX
# Cada BDX: clave -> ruta. Los nombres reales tienen irregularidades (espacios dobles, "_ ", Transporte/Transportes): se respetan tal cual.
carpeta_bdx = f"{directorio_proyecto}/Insumos/{AñoMes}"
BDX_PATHS = {
    6:  f"{carpeta_bdx}/Bordereaux_Inbursa_Casco_Pandi_Transporte_09-11_{str1_month}_{ANIO}.xlsx",
    7:  f"{carpeta_bdx}/Bordereaux_Inbursa_Casco_Pandi_Transporte_11-13_{str1_month}_{ANIO}.xlsx",
    8:  f"{carpeta_bdx}/Bordereaux_Inbursa_Casco_Pandi_Transportes_13-15_{str1_month}_{ANIO}.xlsx",
    9:  f"{directorio_proyecto}/Insumos/Legacy/Bordereaux_Inbursa_Aguas Profundas_12-14_{str1_month}_{ANIO}.xlsx",
    10: f"{carpeta_bdx}/Banorte_BDX PEMEX NCGL-070-1000773 - {str1_month} {ANIO}  Casco_Pandi_Transporte y Plataformas.xlsx",
    11: f"{carpeta_bdx}/Bordereaux_Seguros Atlas E01-2-60-3_{str2_month} {ANIO}.xlsx",
    12: f"{carpeta_bdx}/Banorte_BDX Pemex NCGL-070-1002258 REAS {str1_month} {ANIO}.xlsx",
    13: f"{carpeta_bdx}/Bordereaux_Seguros Atlas E01-2-60-10_ {str2_month} {ANIO}.xlsx",
    14: f"{carpeta_bdx}/Bordereaux_Mapfre_Casco_Pandi_Transportes_21-23_{str1_month}_{ANIO}.xlsx",
}
LEGACY_KEYS = {9}  # Aguas Profundas: se incluye en la salida con Net Reserve = 0 (no se actualiza)

# Validar que existan todos los archivos antes de leer alguno
faltantes = {k: p for k, p in BDX_PATHS.items() if not os.path.isfile(p)}
if faltantes:
    detalle = "\n".join(f"  - path_bdx_{k}: {p}" for k, p in faltantes.items())
    raise FileNotFoundError(f"Faltan {len(faltantes)} archivo(s) BDX:\n{detalle}")

SCAN_ROWS = 100      # filas iniciales donde se buscan Policy / Local Carrier / KOT (encabezado de la hoja)
KOT_MAX_ROW = 60     # la celda "KOT" debe estar en las primeras 60 filas
POLICY_TARGETS = {"POLICY N° (LOCAL)", "POLICY N°"}

In [26]:
# =============================================================================
# Funciones
# =============================================================================
def _clean(value):
    return str(value).replace("\xa0", " ").strip().upper() if value else ""


def extraer_net_reserve(path_excel, tolerance=1):

    # Sin read_only: se necesita ws.merged_cells para ubicar el rango de la sección KOT
    wb = load_workbook(path_excel, data_only=True)
    resultados = []

    for sheet_name in wb.sheetnames:
        # Excluir hojas "A Cobro" e "Instrucciones": no son pólizas
        if sheet_name.strip().upper() in ("A COBRO", "INSTRUCCIONES"):
            continue

        ws = wb[sheet_name]
        max_row = ws.max_row
        max_col = ws.max_column

        policy = None
        carrier_value = None
        kot_row = kot_col_min = kot_col_max = None

        # -------------------------------------------------
        # Una sola pasada por el encabezado: Policy, Local Carrier y KOT
        # -------------------------------------------------
        for row in ws.iter_rows(min_row=1, max_row=min(SCAN_ROWS, max_row)):
            for cell in row:
                if not cell.value:
                    continue
                cv = _clean(cell.value)

                if policy is None and cv in POLICY_TARGETS and cell.column + 2 <= max_col:
                    policy = ws.cell(row=cell.row, column=cell.column + 2).value
                elif carrier_value is None and cv == "LOCAL CARRIER" and cell.column + 2 <= max_col:
                    carrier_value = ws.cell(row=cell.row, column=cell.column + 2).value
                elif kot_row is None and cv == "KOT" and cell.row <= KOT_MAX_ROW:
                    kot_row = cell.row
                    kot_col_min = kot_col_max = cell.column
                    # Si "KOT" está en una celda combinada, tomar todo el rango de columnas
                    for mr in ws.merged_cells.ranges:
                        if mr.min_row <= cell.row <= mr.max_row and mr.min_col <= cell.column <= mr.max_col:
                            kot_col_min, kot_col_max = mr.min_col, mr.max_col
                            break

        # -------------------------------------------------
        # "NET RESERVE (USD)" dentro del rango de columnas de la sección KOT
        # -------------------------------------------------
        header_row_index = None
        target_col = None

        if kot_row is not None:
            for row in ws.iter_rows(min_row=kot_row, max_row=min(kot_row + 15, max_row)):
                for cell in row:
                    if cell.value and "NET RESERVE (USD)" in _clean(cell.value) \
                            and kot_col_min <= cell.column <= kot_col_max:
                        header_row_index, target_col = cell.row, cell.column
                        break
                if header_row_index:
                    break

        # -------------------------------------------------
        # Fallback: primera fila con "NET RESERVE (USD)" en las primeras 25 filas;
        # si hay varias columnas, se toma la 2a
        # -------------------------------------------------
        if header_row_index is None:
            for row in ws.iter_rows(min_row=1, max_row=25):
                cols = [c.column for c in row if c.value and "NET RESERVE (USD)" in _clean(c.value)]
                if cols:
                    header_row_index = row[0].row
                    target_col = cols[1] if len(cols) >= 2 else cols[0]
                    break

        # -------------------------------------------------
        # Último valor numérico de la columna (la tolerancia se aplica solo al resultado final).
        # Si hay una fila rotulada "TOTAL" y difiere del último valor, se avisa.
        # -------------------------------------------------
        net_reserve_value = None

        if header_row_index is not None and target_col is not None:
            last_num = None
            total_num = None

            for vals in ws.iter_rows(min_row=header_row_index + 1, max_row=max_row,
                                     max_col=target_col, values_only=True):
                val = vals[target_col - 1] if len(vals) >= target_col else None
                if val is None:
                    continue
                try:
                    num = float(str(val).replace(",", "").replace("$", ""))
                except (ValueError, TypeError):
                    print(f"  [aviso] Valor no numérico ignorado en '{sheet_name}': {val!r}")
                    continue
                last_num = num
                if any(isinstance(v, str) and "TOTAL" in v.upper() for v in vals[:target_col - 1]):
                    total_num = num

            if last_num is not None:
                if total_num is not None and abs(total_num - last_num) > 0.01:
                    print(f"  [aviso] '{sheet_name}': fila TOTAL ({total_num:,.2f}) difiere del último valor ({last_num:,.2f}); se usa el último")
                net_reserve_value = 0.0 if abs(last_num) < tolerance else last_num

        resultados.append({
            "Policy Number": policy,
            "Cedant": carrier_value,
            "Cover": sheet_name,
            "Net Reserve": net_reserve_value
        })

    wb.close()
    return pd.DataFrame(resultados)


def extraer_lote(keys, etiqueta):
    frames = []
    for k in keys:
        try:
            df = extraer_net_reserve(BDX_PATHS[k])
            print(f"  path_bdx_{k}: {len(df)} hoja(s) leídas ({etiqueta})")
            frames.append(df)
        except Exception as e:
            raise RuntimeError(f"Error procesando path_bdx_{k} ({etiqueta}): {BDX_PATHS[k]}") from e
    return pd.concat(frames, ignore_index=True)


# RENOMBRAR COVER: gana la primera clave que aparezca en el nombre (el orden importa)
COVER_KEYS = ["TRANSPORTE", "FERREO", "FLETADORES RC", "CASCO", "CARGA", "PANDI"]

def normalizar_cover(x):
    if isinstance(x, str):
        up = x.upper()
        for key in COVER_KEYS:
            if key in up:
                return key
    return x

In [ ]:
# =============================================================================
# Ejecución: extracción, validaciones y salida
# =============================================================================
df_total = extraer_lote([k for k in BDX_PATHS if k not in LEGACY_KEYS], "actual")

# Legacy records: include in output but with Net Reserve = 0
if LEGACY_KEYS:
    df_legacy = extraer_lote(sorted(LEGACY_KEYS), "legacy")
    df_legacy['Net Reserve'] = 0.0
    df_total = pd.concat([df_total, df_legacy], ignore_index=True)

df_total["Cover"] = df_total["Cover"].apply(normalizar_cover)
df_total["Net Reserve"] = df_total["Net Reserve"].fillna(0)

# ------------------------- Validaciones (avisos, no bloquean) -------------------------
if df_total.empty:
    raise ValueError("No se extrajo ninguna póliza de los BDX")

print(f"\nTotal filas: {len(df_total)} | Net Reserve total: {df_total['Net Reserve'].sum():,.2f}")

for col in ("Policy Number", "Cedant"):
    vacios = df_total[df_total[col].isna() | (df_total[col].astype(str).str.strip() == "")]
    if not vacios.empty:
        print(f"[aviso] {len(vacios)} fila(s) sin '{col}':")
        print(vacios[["Policy Number", "Cedant", "Cover", "Net Reserve"]].to_string())

duplicados = df_total[df_total.duplicated(subset=["Policy Number", "Cover"], keep=False)
                      & df_total["Policy Number"].notna()]
if not duplicados.empty:
    print(f"[aviso] {len(duplicados)} fila(s) con Policy Number + Cover duplicados:")
    print(duplicados.to_string())

# Comparación contra el OSLR del periodo anterior
ruta_ant = f"{directorio_proyecto2}/Contabilidad/{AñoMes_ant}/OSLR {AñoMes_ant}.xlsx"
if os.path.isfile(ruta_ant):
    df_ant = pd.read_excel(ruta_ant)
    print(f"\nVs OSLR {AñoMes_ant}: filas {len(df_ant)} -> {len(df_total)} | "
          f"Net Reserve {df_ant['Net Reserve'].sum():,.2f} -> {df_total['Net Reserve'].sum():,.2f}")
    llave = ["Policy Number", "Cover"]
    ant_keys = set(map(tuple, df_ant[llave].astype(str).values))
    act_keys = set(map(tuple, df_total[llave].astype(str).values))
    if ant_keys - act_keys:
        print(f"[aviso] Pólizas que estaban en {AñoMes_ant} y ya no aparecen: {sorted(ant_keys - act_keys)}")
    if act_keys - ant_keys:
        print(f"[info] Pólizas nuevas vs {AñoMes_ant}: {sorted(act_keys - ant_keys)}")
else:
    print(f"[aviso] No existe {ruta_ant}; se omite la comparación con el periodo anterior")

# ------------------------- Salida -------------------------
ruta = f"{directorio_proyecto2}/Contabilidad/{AñoMes}"
os.makedirs(ruta, exist_ok=True)
salida = f"{ruta}/OSLR {AñoMes_Output or AñoMes}.xlsx"
df_total.to_excel(salida, index=False)
print(f"\nArchivo generado: {salida}")

  path_bdx_6: 3 hoja(s) leídas (actual)
  path_bdx_7: 3 hoja(s) leídas (actual)
  path_bdx_8: 3 hoja(s) leídas (actual)
  path_bdx_10: 4 hoja(s) leídas (actual)
  path_bdx_11: 5 hoja(s) leídas (actual)
  path_bdx_12: 8 hoja(s) leídas (actual)
  [aviso] Valor no numérico ignorado en 'Ferreo - Polietileno': 'Reserva Neta-USD'
  [aviso] Valor no numérico ignorado en 'Ferreo - Polietileno': '#REF!'
  [aviso] Valor no numérico ignorado en 'Reparadores de Buques': 'Reserva Neta-USD'
  [aviso] Valor no numérico ignorado en 'Remesas y existencias de D': 'Reserva Neta-USD'
  [aviso] Valor no numérico ignorado en 'Ant.y obras de arte': 'Reserva Neta-USD'
  path_bdx_13: 9 hoja(s) leídas (actual)
  path_bdx_14: 5 hoja(s) leídas (actual)
  path_bdx_9: 1 hoja(s) leídas (legacy)

Total filas: 41 | Net Reserve total: 4,180,825.57
[aviso] 2 fila(s) sin 'Policy Number':
   Policy Number Cedant   Cover    Net Reserve
28           NaN    NaN   D Y V  618994.392036
29           NaN    NaN  FERREO       0.0